In [ ]:
#%% PACKAGES
# Basics
import numpy as np                                                          
from math import pi                                                            
import warnings                                                              
import os    

from datetime import datetime, timedelta
from pandas import DataFrame                                                                 

# Visualization
import pandas as pd                                                          
import matplotlib.pyplot as plt                                                
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import numpy as np
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator

In [ ]:
#Set the working directory
os.chdir('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Notebook')

#Read the files
IndexRaw = pd.read_csv('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/step2_features.csv')

In [ ]:
#Features list: column names from processed data
#[accidents, amenities, ]

In [ ]:
#Fill NULL
Indexv1 = IndexRaw.fillna(0)

list(Indexv1)

#rename column if necessary

#Adding intervals (for factors where we have an interval of interest and below or above that interval the situation doesn't affect the walkability)

Indexv1['amenities'] = Indexv1['amenities'].clip(upper=50) #more than 50 amenities in 350-meters-radius around segment centroid
''' --- other intervals ---
#sidewalks wider than 5 meters
#sidewalks narrower than 0.9 meters
#more than 100% parking pressure
'''

#Remove segments of less than 1m length
print("Keep only segment > 1 meter")
length_min = 1
Indexv2 = Indexv1[Indexv1['length'] > length_min]  



In [ ]:
# Normalising fields
Indexv2 = Indexv2[['segment_id']].copy()

#Rename columns
print("Rename columns, add all the features --> ")
Indexv2['N-RoadSafety'] = Indexv1['accidents']
Indexv2['N-ProxAmenities'] = Indexv1['amenities']



In [ ]:

# Min Max Normalisation
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
Indexv3 = Indexv2

print("Scaler, ensure all features are included: add if necessary -->")
Indexv3[['N-ProxAmenities', 'N-RoadSafety']] = scaler.fit_transform(Indexv3[['N-ProxAmenities', 'N-RoadSafety']])


In [ ]:
# Inverse columns (for factors that have a negative effect on walkability)
Indexv3['N-RoadSafety'] = 1 - Indexv3['N-RoadSafety'] #accidents


In [ ]:
#Check data distribution(use this to check if there are still outliers skewing the factors)

Indexv3[['N-ProxAmenities', 'N-RoadSafety']].plot(kind='box', subplots=True, layout=(1, 2), figsize=(10, 4))
plt.tight_layout()
plt.show()


In [ ]:
# Calculate the Main Index Scores
Indexv4 = Indexv3

print('Modify weights for main index ->')

Zscore_weights = {
    'N-RoadSafety': 0.7,
    'N-ProxAmenities': 0.3}

# Function to calculate sub-index score
def calculate_index(df, weights_dict):
    return sum(df[col] * weight for col, weight in weights_dict.items())

Indexv4['I-Zscore'] = calculate_index(Indexv4, Zscore_weights)


#Normalising Index
Indexv4['Non-Scaled']=Indexv4['I-Zscore']
Indexv4[['I-Zscore']] = scaler.fit_transform(Indexv4[['I-Zscore']])



In [ ]:

#%% SUB-INDEXES
# Define weights for each sub-index in dictionaries
'''
print('Modify weights for sub-indexes ->>')
feasibility_weights = {
    'N-Furniture': 0.056,
    'N-Green': 0.048,
    'N-ParksPlaza': 0.056,
    'N-ParkinPres': 0.059,
    'N-ProxAmenities': 0.053,
}

accessibility_weights = {
    'N-RoadSafety': 0.094,
    'N-MaxSpeed': 0.073,
}

safety_weights = {
    'N-ProxAmenit': 0.073,
    'N-ShortBlocks': 0.031,
    'N-OV': 0.068,
}

comfort_weights = {
    'N-ProxAmenities': 0.059,
    'N-Crime': 0.071,
    'N-Lighting': 0.074,
}

pleasurability_weights = {
    'N-Obstacles': 0.093,
    'N-SidewalkWi': 0.086,
    'N-Maintenance': 0.065,
}


# Apply to Indexv4 and store results in Indexv5
Indexv5 = Indexv4.copy()
Indexv5['Feasibility'] = calculate_index(Indexv4, feasibility_weights)
Indexv5['Accessibility'] = calculate_index(Indexv4, accessibility_weights)
Indexv5['Safety'] = calculate_index(Indexv4, safety_weights)
Indexv5['Comfort'] = calculate_index(Indexv4, comfort_weights)
Indexv5['Pleasurability'] = calculate_index(Indexv4, pleasurability_weights)


# Min Max Normalisation
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
Indexv5[['Feasability','Accessibility','Safety','Comfort','Pleasurability']] = scaler.fit_transform(Indexv5[['Feasability','Accessibility','Safety','Comfort','Pleasurability']])

'''

In [ ]:
# Save it
path = "/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-3"
os.chdir(path)
Indexv4.to_csv('Index-Walkability.csv', index = False)

#After saving, perform a join-by-field value in QGIS with the shapefiles of the street segments. Join using the fields called "ID"

In [ ]:
Indexv4.head(50)